In [1]:
from pathlib import Path
import re
import os
from tqdm.auto import tqdm

YEAR = 2003

# Point this to your year folder
YEAR_DIR = Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\reforecast\version_4_0\consolidated\discharge\grib2\area_35_63_4_131") / str(YEAR)

EXTRACT_BASE = YEAR_DIR / "_staging" / "extract"     # contains 2003_01, 2003_02, ...
OUT_PATH = YEAR_DIR / "data_year_repacked.grib"      # write to a NEW file first

assert EXTRACT_BASE.exists(), f"Missing: {EXTRACT_BASE}"

def mb(p: Path) -> float:
    return p.stat().st_size / (1024 * 1024)

def discover_monthly_gribs(extract_base: Path, year: int) -> list[Path]:
    # expected folder: YYYY_MM
    pat = re.compile(rf"^{year}_(\d{{2}})$")
    month_dirs = []
    for d in extract_base.iterdir():
        if d.is_dir():
            m = pat.match(d.name)
            if m:
                month_dirs.append((int(m.group(1)), d))
    month_dirs.sort(key=lambda x: x[0])

    files = []
    for mm, d in month_dirs:
        # You said inside each month folder there's a data.grib
        f = d / "data.grib"
        if f.exists() and f.stat().st_size > 0:
            files.append(f)
        else:
            print(f"⚠ Missing/empty monthly file for {year}_{mm:02d}: {f}")
    return files

monthly_files = discover_monthly_gribs(EXTRACT_BASE, YEAR)
print("Monthly files found:", len(monthly_files))
for f in monthly_files[:3]:
    print(" ", f, f"({mb(f):.1f} MB)")
print("Total monthly size (GB):", sum(f.stat().st_size for f in monthly_files)/1e9)

Monthly files found: 6
  C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\reforecast\version_4_0\consolidated\discharge\grib2\area_35_63_4_131\2003\_staging\extract\2003_03\data.grib (1182.5 MB)
  C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\reforecast\version_4_0\consolidated\discharge\grib2\area_35_63_4_131\2003\_staging\extract\2003_04\data.grib (4730.1 MB)
  C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\reforecast\version_4_0\consolidated\discharge\grib2\area_35_63_4_131\2003\_staging\extract\2003_05\data.grib (5321.4 MB)
Total monthly size (GB): 28.519447868


In [2]:
def starts_with_grib(p: Path) -> bool:
    try:
        with open(p, "rb") as f:
            return f.read(4) == b"GRIB"
    except Exception:
        return False

bad = [p for p in monthly_files if not starts_with_grib(p)]
print("Monthly files not starting with GRIB:", len(bad))
if bad:
    for p in bad[:10]:
        print(" ", p)

Monthly files not starting with GRIB: 0


In [3]:
from eccodes import (
    codes_grib_new_from_file,
    codes_get_message,
    codes_release,
)

def repack_yearly_grib(in_files: list[Path], out_path: Path) -> dict:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        out_path.unlink()  # ensure clean output

    total_msgs = 0
    per_file = {}

    with open(out_path, "wb") as w:
        for fp in tqdm(in_files, desc="Repacking months", leave=True):
            n = 0
            with open(fp, "rb") as f:
                while True:
                    gid = codes_grib_new_from_file(f)
                    if gid is None:
                        break
                    try:
                        msg = codes_get_message(gid)  # raw bytes for THIS message
                        w.write(msg)
                        n += 1
                    finally:
                        codes_release(gid)

            per_file[str(fp)] = n
            total_msgs += n

    return {"total_msgs": total_msgs, "per_file": per_file, "out_path": str(out_path)}

info = repack_yearly_grib(monthly_files, OUT_PATH)
print("Wrote:", info["out_path"])
print("Total messages:", info["total_msgs"])
print("Output size (GB):", OUT_PATH.stat().st_size / 1e9)

Repacking months:   0%|          | 0/6 [00:00<?, ?it/s]

OSError: [Errno 28] No space left on device

In [16]:
from eccodes import codes_grib_new_from_file, codes_release

def count_messages(fp: Path) -> int:
    n = 0
    with open(fp, "rb") as f:
        while True:
            gid = codes_grib_new_from_file(f)
            if gid is None:
                break
            n += 1
            codes_release(gid)
    return n

n_out = count_messages(YEARLY)
print("Yearly GRIB messages:", n_out)

Yearly GRIB messages: 3680


In [15]:
from pathlib import Path
from collections import Counter
from eccodes import codes_grib_new_from_file, codes_get, codes_is_defined, codes_release

YEARLY = Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\reforecast\version_4_0\consolidated\discharge\grib2\area_35_63_4_131\2003\_staging\extract\2003_04\data.grib")

def yyyymmdd_to_year_month(v: int):
    y = v // 10000
    m = (v // 100) % 100
    d = v % 100
    return y, m, d

def scan_months(grib_path: Path, report_every: int = 200_000):
    init_months = Counter()   # from dataDate
    valid_months = Counter()  # from validityDate
    init_minmax = {}          # month -> [min_date, max_date]
    valid_minmax = {}

    n = 0
    with open(grib_path, "rb") as f:
        while True:
            gid = codes_grib_new_from_file(f)
            if gid is None:
                break
            try:
                # init date (dataDate) == yyyymmdd
                if codes_is_defined(gid, "dataDate"):
                    dd = int(codes_get(gid, "dataDate"))
                    y, m, d = yyyymmdd_to_year_month(dd)
                    init_months[(y, m)] += 1
                    init_minmax.setdefault((y, m), [dd, dd])
                    init_minmax[(y, m)][0] = min(init_minmax[(y, m)][0], dd)
                    init_minmax[(y, m)][1] = max(init_minmax[(y, m)][1], dd)

                # verifying date (validityDate) == yyyymmdd
                if codes_is_defined(gid, "validityDate"):
                    vd = int(codes_get(gid, "validityDate"))
                    y, m, d = yyyymmdd_to_year_month(vd)
                    valid_months[(y, m)] += 1
                    valid_minmax.setdefault((y, m), [vd, vd])
                    valid_minmax[(y, m)][0] = min(valid_minmax[(y, m)][0], vd)
                    valid_minmax[(y, m)][1] = max(valid_minmax[(y, m)][1], vd)

                n += 1
                if report_every and (n % report_every == 0):
                    print(f"...scanned {n:,} messages")
            finally:
                codes_release(gid)

    return n, init_months, valid_months, init_minmax, valid_minmax

n, init_months, valid_months, init_minmax, valid_minmax = scan_months(YEARLY)

print("\nTotal messages:", f"{n:,}")

print("\n=== INIT months present (from dataDate) ===")
for (y, m) in sorted(init_months):
    mn, mx = init_minmax[(y, m)]
    print(f"{y}-{m:02d}: {init_months[(y,m)]:,} msgs | dataDate range {mn}–{mx}")

print("\n=== VALID months present (from validityDate) ===")
for (y, m) in sorted(valid_months):
    mn, mx = valid_minmax[(y, m)]
    print(f"{y}-{m:02d}: {valid_months[(y,m)]:,} msgs | validityDate range {mn}–{mx}")

# Manual completeness check for init months inside the target year
expected = {(2003, m) for m in range(1, 13)}
present = set(init_months.keys())

missing = sorted(expected - present)
extra = sorted(present - expected)

print("\nExpected init months:", sorted(expected))
print("Missing init months :", missing)
print("Extra init months   :", extra)


Total messages: 3,680

=== INIT months present (from dataDate) ===
2003-04: 3,680 msgs | dataDate range 20030403–20030427

=== VALID months present (from validityDate) ===
2003-04: 1,200 msgs | validityDate range 20030404–20030430
2003-05: 2,200 msgs | validityDate range 20030501–20030531
2003-06: 280 msgs | validityDate range 20030601–20030612

Expected init months: [(2003, 1), (2003, 2), (2003, 3), (2003, 4), (2003, 5), (2003, 6), (2003, 7), (2003, 8), (2003, 9), (2003, 10), (2003, 11), (2003, 12)]
Missing init months : [(2003, 1), (2003, 2), (2003, 3), (2003, 5), (2003, 6), (2003, 7), (2003, 8), (2003, 9), (2003, 10), (2003, 11), (2003, 12)]
Extra init months   : []


In [ ]:
import cfgrib
import numpy as np

# Write cfgrib index next to OUT_PATH, but separate file name
idx_path = str(OUT_PATH) + ".idx"

dsets = cfgrib.open_datasets(str(OUT_PATH), indexpath=idx_path, errors="ignore")
print("Datasets:", len(dsets))
ds = dsets[0]
print("dims:", dict(ds.dims))
print("coords:", list(ds.coords))
print("data_vars:", list(ds.data_vars))

# Touch a small value to force a read
var = "dis24" if "dis24" in ds.data_vars else list(ds.data_vars)[0]
x = ds[var].isel(step=0).isel(number=0).isel(time=0).isel(latitude=0).isel(longitude=0).values
print("Sample value:", float(x))